# ESM-2 embedding extraction (free Colab GPU)

**Runtime → Change runtime type → T4 GPU** before running.

This notebook:
1. installs `transformers`,
2. lets you upload `train.csv`, `val.csv`, `test.csv` from `data/processed/`,
3. mean-pools ESM-2 last-hidden-state embeddings over the real residue tokens (CLS/EOS excluded),
4. saves `{model_tag}_{split}_{data_hash}.npy` caches and zips them for download.

The `data_hash` and filenames are computed **exactly** as in `src/encoders.py` /
`src/data_prep.py`, so after you unzip the downloaded files into `embeddings_cache/`
in the repo, `python -m src.train --model esm2` will load them instead of recomputing.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
!pip -q install --upgrade transformers

In [ ]:
# ESM-2 small (default) or the 35M variant
MODEL_ID = "facebook/esm2_t6_8M_UR50D"      # or "facebook/esm2_t12_35M_UR50D"
BATCH_SIZE = 32
MAX_LEN = 50 + 2                             # peptide length cap (+ BOS/EOS)
MODEL_TAG = MODEL_ID.rsplit("/", 1)[-1]
print("model:", MODEL_ID, "| tag:", MODEL_TAG)

In [ ]:
# Upload train.csv, val.csv, test.csv from your local data/processed/ folder.
import os
from google.colab import files
os.makedirs("data/processed", exist_ok=True)
print("Select train.csv, val.csv and test.csv ...")
uploaded = files.upload()
for name, content in uploaded.items():
    with open(os.path.join("data/processed", os.path.basename(name)), "wb") as f:
        f.write(content)
print("saved:", sorted(os.listdir("data/processed")))

In [ ]:
# data_hash: sha256 of train+val+test CSV bytes, first 12 hex chars (matches the repo).
import hashlib
def data_hash_12(paths):
    h = hashlib.sha256()
    for p in paths:
        with open(p, "rb") as fh:
            h.update(fh.read())
    return h.hexdigest()[:12]

SPLIT_PATHS = {s: f"data/processed/{s}.csv" for s in ("train", "val", "test")}
DATA_HASH = data_hash_12([SPLIT_PATHS["train"], SPLIT_PATHS["val"], SPLIT_PATHS["test"]])
print("data_hash:", DATA_HASH, "(must match split_stats.json on your laptop)")

In [ ]:
# Mean-pooled ESM-2 embeddings (special tokens + padding excluded).
import numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).to(device).eval()

@torch.no_grad()
def extract(sequences, batch_size=BATCH_SIZE):
    seqs = [str(s).strip().upper() for s in sequences]
    chunks = []
    for i in range(0, len(seqs), batch_size):
        batch = seqs[i:i + batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True,
                        max_length=MAX_LEN, return_special_tokens_mask=True)
        special = enc.pop("special_tokens_mask").to(device)
        enc = {k: v.to(device) for k, v in enc.items()}
        hidden = model(**enc).last_hidden_state
        residue = (enc["attention_mask"] * (1 - special)).unsqueeze(-1).float()
        mean = (hidden * residue).sum(1) / residue.sum(1).clamp(min=1.0)
        chunks.append(mean.cpu().numpy().astype(np.float32))
    return np.concatenate(chunks, axis=0)

In [ ]:
# Extract for each split, save the .npy caches, then zip + download.
import time, glob
for split, path in SPLIT_PATHS.items():
    df = pd.read_csv(path)
    t0 = time.time()
    emb = extract(df["sequence"].tolist())
    out = f"{MODEL_TAG}_{split}_{DATA_HASH}.npy"
    np.save(out, emb)
    print(f"{split:5s}: {emb.shape} -> {out}  ({time.time() - t0:.1f}s)")

npys = sorted(glob.glob(f"{MODEL_TAG}_*_{DATA_HASH}.npy"))
zipname = f"esm_embeddings_{MODEL_TAG}_{DATA_HASH}.zip"
!zip -q {zipname} {' '.join(npys)}
from google.colab import files as _files
_files.download(zipname)
print("Download:", zipname, "— unzip into embeddings_cache/ on your laptop")